# Controlled historical-context ablation — mean-reverting synthetic market

K0, K1, K5 and K60 expose zero, one, five or sixty historical observations to the
**same** Vanilla MLP diffusion encoder. Every input still has shape `[batch, 60, 12]`.
Earlier **standardized** observations are zeroed (the training mean), without changing
raw histories, targets, scaler, dates, architecture or parameter count.

The common predeclared budget is **15 epochs**, with best-validation-loss selection.
Only that budget differs from `synthetic_mean_reverting.yaml`. This is a diagnostic
budget, not a claim of optimality. The historical 50-epoch K60 run is excluded.

**Hypotheses, not outcomes:** K0 may show little direct lag dependence; K1 may help
for a one-lag process; K5 tests modest additional history; K60 may introduce optimization
difficulty or overfitting. Rank forecasting quality using CRPS, Energy Score and
predictive log-mean MSE, not lag slope alone. Validation selected the checkpoints:
comparisons and block intervals are exploratory development evidence, not independent
held-out or selection-adjusted significance. The test partition stays untouched.

Run All performs **only two epochs each for K1 and K60**, followed by first-64-date
inference (256 scenarios). All four 15-epoch commands are Markdown at the end.

## Setup

Enable Kaggle Internet; a GPU is useful for inference. This uses the established
`feature/synthetic-market` synchronization cell. No existing artifacts are restored
or modified automatically.

In [ ]:
import os
import sys

REPO_PATH = "/kaggle/working/diffusion-portfolio"
USERNAME = "Amir-Entezari"
REPO = "diffusion-portfolio"
BRANCH = "feature/synthetic-market"

repo_url = f"https://github.com/{USERNAME}/{REPO}.git"

# Clone the specified branch if the repository doesn't exist.
if not os.path.exists(os.path.join(REPO_PATH, ".git")):
    !git clone --branch {BRANCH} --single-branch {repo_url} {REPO_PATH}

else:
    # Fetch latest changes from the specified branch.
    !git -C {REPO_PATH} fetch origin {BRANCH}

    # Switch to the branch and force-sync with remote.
    !git -C {REPO_PATH} checkout -B {BRANCH} FETCH_HEAD
    !git -C {REPO_PATH} reset --hard FETCH_HEAD

os.chdir(REPO_PATH)

SRC_PATH = os.path.join(REPO_PATH, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# Verify the active branch and latest commit.
!git branch --show-current
!git log -1 --oneline

print(f"Repository synced with origin/{BRANCH}.")

In [ ]:
!pip install -q -e '.[dev]'
assert _exit_code == 0, "Dependency installation failed."

## Focused regression tests

Tests use small CPU fixtures; no full research training or held-out scoring.

In [ ]:
!pytest -q tests/unit/test_history_ablation.py tests/unit/test_synthetic_diffusion_comparison.py tests/unit/test_diffusion_training.py
assert _exit_code == 0, "Context-ablation regression tests failed."

## Two-epoch execution pilot — K1

Each run records its immutable mask and full 15-epoch contract before training.
`--stop-after-epoch 2` is an absolute boundary. A K1 directory cannot resume as
K5, K60 or an unspecified policy. `--resume` restores the existing checkpoint/RNG.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/train_diffusion.py \
    --config configs/ablations/synthetic_ar1_context.yaml --history-keep-last 1 \
    --output-dir /kaggle/working/synthetic_context/k1 --device auto --resume --stop-after-epoch 2
assert _exit_code == 0, "K1 pilot training failed."

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import matplotlib.pyplot as plt

K1_DIR = Path("/kaggle/working/synthetic_context/k1")
history = pd.DataFrame(torch.load(K1_DIR / "last.pt", map_location="cpu", weights_only=False)["history"])
display(history)
ax = history.plot(x="epoch", y=["train_loss", "val_loss"], marker="o", title="Mean-reverting Vanilla K1 — current progress")
ax.set(xlabel="Epoch", ylabel="Diffusion loss")
ax.legend(["Train Loss", "Validation Loss"])
ax.axvline(history.loc[history.val_loss.idxmin(), "epoch"], color="gray", linestyle="--", label="Best validation epoch")
ax.grid(alpha=.3)
plt.show()

## Two-epoch execution pilot — K60

Same configuration and initialization seed, separate checkpoint directory.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/train_diffusion.py \
    --config configs/ablations/synthetic_ar1_context.yaml --history-keep-last 60 \
    --output-dir /kaggle/working/synthetic_context/k60 --device auto --resume --stop-after-epoch 2
assert _exit_code == 0, "K60 pilot training failed."

In [ ]:
K60_DIR = Path("/kaggle/working/synthetic_context/k60")
history = pd.DataFrame(torch.load(K60_DIR / "last.pt", map_location="cpu", weights_only=False)["history"])
display(history)
ax = history.plot(x="epoch", y=["train_loss", "val_loss"], marker="o", title="Mean-reverting Vanilla K60 — current progress")
ax.set(xlabel="Epoch", ylabel="Diffusion loss")
ax.legend(["Train Loss", "Validation Loss"])
ax.axvline(history.loc[history.val_loss.idxmin(), "epoch"], color="gray", linestyle="--")
ax.grid(alpha=.3)
plt.show()

## Verify both checkpoint manifests

Require matching source/scaler, original policy, checkpoint byte bindings, history
and best-checkpoint metadata. Partial runs remain **preliminary**, not completed studies.

In [ ]:
from diffusion_portfolio.config import load_config
from diffusion_portfolio.data.preparation import load_standardizer
from diffusion_portfolio.evaluation.forecast_identity import verify_synthetic_checkpoint

for run_dir in (K1_DIR, K60_DIR):
    cfg = load_config(run_dir / "config.yaml")
    best = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=False)
    manifest = verify_synthetic_checkpoint(run_dir, cfg, load_standardizer(run_dir / "standardizer.npz"), best)
    print(run_dir.name, manifest["training_contract"]["history_mask"], manifest["training_run"])

## Limited validation inference

The evaluator automatically restores each original mask from the immutable contract;
there is no inference mask override. Only the first 64 validation windows enter the
DataLoader. Destinations must be empty or absent; choose new names for repeated
inference or subsequent completed runs. Batch size/device affect individual draws.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluate_diffusion.py \
    --run-dir /kaggle/working/synthetic_context/k1 \
    --split validation --limit 64 --batch-size 16 --device auto \
    --output-dir /kaggle/working/synthetic_context/k1_pilot_inference
assert _exit_code == 0, "K1 limited inference failed."

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluate_diffusion.py \
    --run-dir /kaggle/working/synthetic_context/k60 \
    --split validation --limit 64 --batch-size 16 --device auto \
    --output-dir /kaggle/working/synthetic_context/k60_pilot_inference
assert _exit_code == 0, "K60 limited inference failed."

## Matched observed-only statistical forecasts

Set `BASELINE_DIR` to existing full mean-reverting Stage 4E-1 artifacts with 256 scenarios.
They need not share diffusion's 15-epoch training config, only the exact **data** identity,
assets, dates, raw observations and ensemble size. No mask is applied to statistical models.
If unavailable, leave `None` to generate a small matched baseline. Never substitute KF12
or differently seeded synthetic data. Incompatible archives fail strict comparison.

In [ ]:
BASELINE_DIR = None  # Optional: Path("/your/saved/mean_reverting_full_baselines")
if BASELINE_DIR is None:
    BASELINE_DIR = Path("/kaggle/working/synthetic_context/matched_baseline_pilot")
    !PYTHONUNBUFFERED=1 python -u scripts/benchmark_synthetic_forecasts.py --config configs/ablations/synthetic_ar1_context.yaml --models gaussian,ar1,ewma --split validation --n-scenarios 256 --limit 64 --seed 42 --output-dir {BASELINE_DIR}
    assert _exit_code == 0, "Matched statistical benchmark failed."
else:
    BASELINE_DIR = Path(BASELINE_DIR)
    assert (BASELINE_DIR / "report.json").is_file(), "Missing existing baseline report."

## Paired comparisons against statistical baselines

The existing tool verifies full source alignment and uses the same 64 targets from
longer baseline archives. Negative diffusion-minus-baseline differences favor diffusion.
Block length 20 and 1,000 replications are fixed here; 64-date intervals are exploratory.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/compare_synthetic_forecasts.py \
    --baseline-dir {BASELINE_DIR} \
    --diffusion-dir /kaggle/working/synthetic_context/k1_pilot_inference \
    --block-length 20 --n-bootstrap 1000 --bootstrap-seed 42 \
    --output-dir /kaggle/working/synthetic_context/k1_pilot_comparison
assert _exit_code == 0, "K1 paired baseline comparison failed."

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/compare_synthetic_forecasts.py \
    --baseline-dir {BASELINE_DIR} \
    --diffusion-dir /kaggle/working/synthetic_context/k60_pilot_inference \
    --block-length 20 --n-bootstrap 1000 --bootstrap-seed 42 \
    --output-dir /kaggle/working/synthetic_context/k60_pilot_comparison
assert _exit_code == 0, "K60 paired baseline comparison failed."

## Context and conditional-dependence diagnostics

The archive-only diagnostic uses **actual unmasked raw history**, converts with `log1p`,
averages log scenarios and demeans separately per asset before pooling the lag slope.
The true coefficient 0.4 is a reference only. Forecast-ensemble means are noisy estimates,
not exact conditional expectations. Scores use the existing arithmetic-return metrics.

Direct arm comparisons require exactly equal dates, observations, scenario counts,
configuration except mask, and completed budgets. No date intersection is silently taken.
Paired block intervals condition on saved ensembles and omit training, model-selection
and repeated Monte Carlo uncertainty.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/diagnose_history_ablation.py \
    --baseline-dir {BASELINE_DIR} \
    --diffusion-dirs K1=/kaggle/working/synthetic_context/k1_pilot_inference K60=/kaggle/working/synthetic_context/k60_pilot_inference \
    --block-length 20 --n-bootstrap 1000 --bootstrap-seed 42 \
    --output-dir /kaggle/working/synthetic_context/pilot_diagnostics
assert _exit_code == 0, "Context diagnostics failed."

In [ ]:
import json

DIAGNOSTICS_DIR = Path("/kaggle/working/synthetic_context/pilot_diagnostics")
report = json.loads((DIAGNOSTICS_DIR / "diagnostics.json").read_text())
print(report["interpretation"])
print("Generator AR coefficient (reference only):", report["generator_ar_coefficient_reference_only"])
summary = pd.read_csv(DIAGNOSTICS_DIR / "summary.csv")
display(summary)
display(pd.read_csv(DIAGNOSTICS_DIR / "calibration.csv"))
display(pd.read_csv(DIAGNOSTICS_DIR / "paired_context.csv"))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, metric in zip(axes, ("crps", "energy_score", "log_mean_mse")):
    summary.plot.bar(x="model", y=metric, ax=ax, legend=False, title=metric)
    ax.grid(axis="y", alpha=.3)
fig.tight_layout()
plt.show()

## Matched 15-epoch study — manual commands only

These commands are Markdown so Run All never launches all four budgets. K1/K60
continue from their pilots in the **same** directories and unchanged configuration.
K0/K5 start separately. Best checkpoints are selected within each common 15-epoch budget.

```bash
python -u scripts/train_diffusion.py --config configs/ablations/synthetic_ar1_context.yaml --history-keep-last 0 --output-dir /kaggle/working/synthetic_context/k0 --device cuda --resume
python -u scripts/train_diffusion.py --config configs/ablations/synthetic_ar1_context.yaml --history-keep-last 1 --output-dir /kaggle/working/synthetic_context/k1 --device cuda --resume
python -u scripts/train_diffusion.py --config configs/ablations/synthetic_ar1_context.yaml --history-keep-last 5 --output-dir /kaggle/working/synthetic_context/k5 --device cuda --resume
python -u scripts/train_diffusion.py --config configs/ablations/synthetic_ar1_context.yaml --history-keep-last 60 --output-dir /kaggle/working/synthetic_context/k60 --device cuda --resume
```

After **all four** budgets complete, evaluate each using `evaluate_diffusion.py --split validation`
with a distinct new `--output-dir` (omit `--limit` for all 1,500 validation dates).
Reuse compatible full statistical artifacts or generate matched ones with 256 scenarios.
Run `compare_synthetic_forecasts.py` separately for every arm, then run
`diagnose_history_ablation.py --baseline-dir <baseline> --diffusion-dirs K0=<inference> K1=<inference> K5=<inference> K60=<inference> --output-dir <new-diagnostic-directory>`.
Do not mix pilot and completed budgets or the historical 50-epoch run.

Preserve each run's `best.pt`, `last.pt`, `history.csv`, `config.yaml`, `standardizer.npz`,
`data_identity.json`, `experiment.json` and final `summary.json` when present.
Save inference metadata/archives, baseline archives and diagnostic CSV/JSON outputs with
the Kaggle version. There are no automatic uploads or checkpoint repairs.